In [0]:
# # Spark optimizations for large MERGE operations
spark.conf.set("spark.sql.shuffle.partitions", "auto")
spark.conf.set("spark.databricks.photon.sort.enabled", "true")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.databricks.adaptive.autoOptimizeShuffle.enabled", "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")

In [0]:
# Optional: pass a single hour to process, or leave empty to process all from violations table
dbutils.widgets.text("session_start_hour", "", "Session Start Hour (leave empty to process all violated hours)")
dbutils.widgets.text("max_parallel", "5", "Max parallel hours to process concurrently")

In [0]:
# Column character limits for prod.detection.viewing_content_golden
column_limits = {
    "tvid": 250,
    "zipcode": 10,
    "dma": 128,
    "tms_episode_id": 256,
    "tivo_episode_id": 256,
    "tms_title": 256,
    "tivo_title": 256,
    "tms_airdate": 40,
    "tivo_airdate": 40,
    "tms_channel_callsign": 30,
    "tivo_channel_callsign": 30,
    "tms_channel_affiliate": 40,
    "tivo_channel_affiliate": 40,
    "is_live": 1,
    "ip_address": 32,
    "reported_input_source": 10,
    "content_type": 10,
    "tuner_content_type": 10,
    "input_category": 16,
    "input_device": 32,
    "app_service": 32,
    "tuner_tms_episode_id": 256,
    "tuner_tivo_episode_id": 256,
    "tuner_tms_title": 256,
    "tuner_tivo_title": 256,
    "tuner_tms_airdate": 40,
    "tuner_tivo_airdate": 40,
    "tuner_tms_channel_callsign": 30,
    "tuner_tivo_channel_callsign": 30,
    "tuner_tms_channel_affiliate": 40,
    "tuner_tivo_channel_affiliate": 40,
    "tuner_is_live": 1,
    "tuner_input_category": 16,
    "tuner_input_device": 32,
    "tuner_app_service": 32,
    "enableaudioacr": 1,
    "vizio_epg_channel_id": 32,
    "vizio_epg_program_id": 32,
    "tms_show_genre": 256,
    "tivo_show_genre": 256,
    "tuner_tms_show_genre": 256,
    "tuner_tivo_show_genre": 256,
    "tms_epi_title": 256,
    "tivo_epi_title": 256,
    "tuner_tms_epi_title": 256,
    "tuner_tivo_epi_title": 256,
    "series_id": 256,
    "tuner_series_id": 256,
}

In [0]:
from datetime import datetime

session_hour_param = dbutils.widgets.get("session_start_hour").strip()

# If a specific hour is passed, process only that one; otherwise load all from violations table
if session_hour_param:
    violations_pdf = spark.sql(f"""
        SELECT * FROM dev.public.viewing_content_golden_char_violations
        WHERE session_start_hour >= '2025-10-27 10:00:00'
          AND session_start_hour < '2025-10-27 17:00:00'
        ORDER BY session_start_hour
    """).toPandas()
else:
    violations_pdf = spark.sql("""
        SELECT * FROM dev.public.viewing_content_golden_char_violations
        WHERE session_start_hour >= '2025-10-27 10:00:00'
          AND session_start_hour < '2025-10-27 17:00:00'
        ORDER BY session_start_hour
    """).toPandas()

if not violations_pdf.empty:
    # Sort columns (except 'session_start_hour') alphabetically, keeping 'session_start_hour' first
    cols = ['session_start_hour'] + sorted([c for c in violations_pdf.columns if c != 'session_start_hour'])
    violations_pdf = violations_pdf[cols]

if violations_pdf.empty:
    msg = f"No violations found{' for ' + session_hour_param if session_hour_param else ''}. Nothing to fix."
    print(msg)
    dbutils.notebook.exit(msg)

print(f"Total hours to process: {len(violations_pdf)}")
print(f"Range: {violations_pdf['session_start_hour'].min()} to {violations_pdf['session_start_hour'].max()}")

In [0]:
from concurrent.futures import ThreadPoolExecutor
import threading

max_parallel = int(dbutils.widgets.get("max_parallel"))
total = len(violations_pdf)
fixed_count = 0
failed_hours = []
lock = threading.Lock()

def fix_hour(row):
    """Process a single hour: build and execute the MERGE statement."""
    hour = str(row['session_start_hour'])
    violated_cols = [col_name for col_name in column_limits if row[col_name] == 1]

    if not violated_cols:
        return hour, "skipped", None

    set_clauses = [f"`{col_name}` = SUBSTRING(CAST(`{col_name}` AS STRING), 1, {column_limits[col_name]})" for col_name in violated_cols]

    # Use a minimal USING clause since violations_pdf already identifies the hours to update
    merge_sql = f"""
    MERGE INTO prod.detection.viewing_content_golden AS tgt
    USING (SELECT '{hour}' AS session_start_hour) AS src
    ON tgt.session_start_hour = src.session_start_hour
    WHEN MATCHED THEN
      UPDATE SET {', '.join(set_clauses)}
    """

    try:
        spark.sql(merge_sql)
        return hour, "success", len(violated_cols)
    except Exception as e:
        return hour, "failed", str(e)

print(f"Processing {total} hours with max {max_parallel} parallel threads...")

# Process in order by session_start_hour
ordered_rows = violations_pdf.sort_values("session_start_hour").iterrows()
with ThreadPoolExecutor(max_workers=max_parallel) as executor:
    futures = []
    for i, row in ordered_rows:
        futures.append(executor.submit(fix_hour, row))

    for future in futures:
        hour, status, detail = future.result()
        with lock:
            if status == "success":
                fixed_count += 1
            elif status == "failed":
                failed_hours.append((hour, detail))

            if (fixed_count + len(failed_hours)) % 100 == 0 or (fixed_count + len(failed_hours)) == total:
                print(f"  Progress: {fixed_count} fixed, {len(failed_hours)} failed, {total - fixed_count - len(failed_hours)} remaining ({datetime.now().strftime('%H:%M:%S')})")

print(f"\nComplete! Fixed: {fixed_count}, Failed: {len(failed_hours)}, Total: {total}")
if failed_hours:
    print(f"Failed hours:")
    for h, err in failed_hours:
        print(f"{h}: {err}")

msg = f"Done! Fixed {fixed_count}/{total} hours. Failures: {len(failed_hours)}"
dbutils.notebook.exit(msg)